# Husky Nav2 AI model comparison

This notebook converts the five successful single-DJI runs into an imitation-learning dataset. The target is the velocity command produced by Nav2 (`/cmd_vel_nav`); inputs are the Husky safety scan, goal direction derived from `/plan`, and current odometry velocity.

The split is deliberately **run-based**: runs 1–3 train, run 4 validates model choice, and run 5 is touched only for the final test. Randomly splitting individual timestamps would leak almost-identical neighbouring samples and produce misleading accuracy.

> PPO, SAC and TD3 are reinforcement-learning algorithms and cannot be trained correctly from these static bags alone. The models below establish supervised navigation-policy baselines. The winning network can later initialize an SAC/PPO policy inside an interactive Baylands environment, with the deterministic collision monitor retained as a safety shield.

In [ ]:
from pathlib import Path
import math, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import rosbag2_py
from rclpy.serialization import deserialize_message
from rosidl_runtime_py.utilities import get_message

torch.manual_seed(7)
np.random.seed(7)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
VASTERIVK_ROOT = Path.cwd()
if VASTERIVK_ROOT.name == 'implementation':
    VASTERIVK_ROOT = VASTERIVK_ROOT.parent
IMPLEMENTATION = VASTERIVK_ROOT / 'implementation'
DATASETS = IMPLEMENTATION / 'datasets'
RUN_IDS = [
    'run_20260823_201913',
    'run_20260823_203519',
    'run_20260823_204921',
    'run_20260823_210511',
    'run_20260823_212408',
]
RUNS = [DATASETS / run_id for run_id in RUN_IDS]
assert all((p / 'rosbag/metadata.yaml').exists() for p in RUNS), 'One or more bags are missing'
print('Device:', DEVICE)
print('Runs:', *RUN_IDS, sep='\n  ')

## Feature construction

Each safety scan is compressed into 36 angular sectors using the minimum finite range. This preserves obstacle clearance around the vehicle without feeding hundreds of nearly redundant rays to a small baseline model. The goal is transformed into the Husky body frame. Samples are limited to 2 Hz to reduce temporal duplication.

In [ ]:
def yaw_from_quaternion(q):
    return math.atan2(2*(q.w*q.z + q.x*q.y), 1 - 2*(q.y*q.y + q.z*q.z))

def scan_sectors(msg, sectors=36, cap=12.0):
    values = np.asarray(msg.ranges, dtype=np.float32)
    values[~np.isfinite(values)] = cap
    values = np.clip(values, float(msg.range_min), cap)
    chunks = np.array_split(values, sectors)
    return np.asarray([float(x.min()) if len(x) else cap for x in chunks], dtype=np.float32)

def extract_run(run_dir, sample_hz=2.0, sectors=36):
    bag_dir = run_dir / 'rosbag'
    reader = rosbag2_py.SequentialReader()
    reader.open(
        rosbag2_py.StorageOptions(uri=str(bag_dir), storage_id='mcap'),
        rosbag2_py.ConverterOptions(input_serialization_format='cdr', output_serialization_format='cdr'),
    )
    topic_types = {x.name: x.type for x in reader.get_all_topics_and_types()}
    wanted = {'/husky/safety_scan', '/odom', '/plan', '/cmd_vel_nav'}
    missing = wanted - set(topic_types)
    if missing:
        raise RuntimeError(f'{run_dir.name}: missing {sorted(missing)}')
    classes = {name: get_message(topic_types[name]) for name in wanted}
    odom = plan = command = None
    last_sample_ns = -10**30
    period_ns = int(1e9 / sample_hz)
    rows = []
    while reader.has_next():
        topic, raw, stamp_ns = reader.read_next()
        if topic not in wanted:
            continue
        msg = deserialize_message(raw, classes[topic])
        if topic == '/odom':
            odom = msg
        elif topic == '/plan':
            plan = msg
        elif topic == '/cmd_vel_nav':
            command = msg
        elif stamp_ns - last_sample_ns >= period_ns and odom is not None and command is not None and plan is not None and plan.poses:
            last_sample_ns = stamp_ns
            pose = odom.pose.pose
            yaw = yaw_from_quaternion(pose.orientation)
            goal = plan.poses[-1].pose.position
            dx, dy = goal.x - pose.position.x, goal.y - pose.position.y
            c, s = math.cos(yaw), math.sin(yaw)
            goal_forward = c*dx + s*dy
            goal_left = -s*dx + c*dy
            scan = scan_sectors(msg, sectors=sectors)
            row = {f'scan_{i:02d}': float(v) for i, v in enumerate(scan)}
            row.update({
                'run_id': run_dir.name, 'stamp_ns': int(stamp_ns),
                'goal_forward_m': goal_forward, 'goal_left_m': goal_left,
                'goal_distance_m': math.hypot(dx, dy),
                'current_linear_mps': odom.twist.twist.linear.x,
                'current_angular_rps': odom.twist.twist.angular.z,
                'target_linear_mps': command.linear.x,
                'target_angular_rps': command.angular.z,
                'minimum_clearance_m': float(scan.min()),
            })
            rows.append(row)
    return pd.DataFrame(rows)

cache = VASTERIVK_ROOT / 'nav2_ai_dataset.csv.gz'
if cache.exists():
    data = pd.read_csv(cache)
    print('Loaded cache:', cache)
else:
    frames = []
    for run in RUNS:
        started = time.time()
        frame = extract_run(run)
        frames.append(frame)
        print(run.name, len(frame), 'samples', f'{time.time()-started:.1f}s')
    data = pd.concat(frames, ignore_index=True)
    data.to_csv(cache, index=False, compression='gzip')
    print('Saved cache:', cache)
data.groupby('run_id').size().rename('samples').to_frame()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
data['target_linear_mps'].hist(ax=axes[0], bins=40); axes[0].set_title('Nav2 linear command')
data['target_angular_rps'].hist(ax=axes[1], bins=40); axes[1].set_title('Nav2 angular command')
data['minimum_clearance_m'].hist(ax=axes[2], bins=40); axes[2].set_title('Minimum safety clearance')
for ax in axes: ax.grid(alpha=.2)
plt.tight_layout()

## Leakage-safe train, validation and test split

In [ ]:
TRAIN_RUNS, VAL_RUNS, TEST_RUNS = RUN_IDS[:3], RUN_IDS[3:4], RUN_IDS[4:]
feature_cols = [c for c in data if c.startswith('scan_')] + [
    'goal_forward_m', 'goal_left_m', 'goal_distance_m',
    'current_linear_mps', 'current_angular_rps', 'minimum_clearance_m'
]
target_cols = ['target_linear_mps', 'target_angular_rps']
train = data[data.run_id.isin(TRAIN_RUNS)].copy()
valid = data[data.run_id.isin(VAL_RUNS)].copy()
test = data[data.run_id.isin(TEST_RUNS)].copy()
x_mean = train[feature_cols].mean().to_numpy(np.float32)
x_std = train[feature_cols].std().replace(0, 1).to_numpy(np.float32)
y_mean = train[target_cols].mean().to_numpy(np.float32)
y_std = train[target_cols].std().replace(0, 1).to_numpy(np.float32)
def arrays(frame):
    x = (frame[feature_cols].to_numpy(np.float32)-x_mean)/x_std
    y = (frame[target_cols].to_numpy(np.float32)-y_mean)/y_std
    return x, y
X_train,Y_train=arrays(train); X_val,Y_val=arrays(valid); X_test,Y_test=arrays(test)
pd.DataFrame({'split':['train','validation','test'], 'runs':[TRAIN_RUNS,VAL_RUNS,TEST_RUNS], 'samples':[len(train),len(valid),len(test)]})

In [ ]:
def physical_metrics(pred_scaled, truth_scaled):
    pred = pred_scaled*y_std + y_mean
    truth = truth_scaled*y_std + y_mean
    error = pred-truth
    return {
        'linear_RMSE': float(np.sqrt(np.mean(error[:,0]**2))),
        'angular_RMSE': float(np.sqrt(np.mean(error[:,1]**2))),
        'linear_MAE': float(np.mean(np.abs(error[:,0]))),
        'angular_MAE': float(np.mean(np.abs(error[:,1]))),
    }

results=[]; predictions={}
mean_val=np.zeros_like(Y_val); mean_test=np.zeros_like(Y_test)
results.append({'model':'Mean command','split':'validation',**physical_metrics(mean_val,Y_val)})
results.append({'model':'Mean command','split':'test',**physical_metrics(mean_test,Y_test)})
predictions['Mean command']=mean_test

# Closed-form ridge regression; alpha is selected only on validation data.
ridge_candidates=[]
X1=np.c_[np.ones(len(X_train)),X_train]
V1=np.c_[np.ones(len(X_val)),X_val]
T1=np.c_[np.ones(len(X_test)),X_test]
for alpha in [1e-3,1e-2,1e-1,1,10,100]:
    penalty=np.eye(X1.shape[1],dtype=np.float32)*alpha; penalty[0,0]=0
    weights=np.linalg.solve(X1.T@X1+penalty,X1.T@Y_train)
    score=physical_metrics(V1@weights,Y_val)
    ridge_candidates.append((score['linear_RMSE']+score['angular_RMSE'],alpha,weights,score))
_,ridge_alpha,ridge_weights,ridge_val=min(ridge_candidates,key=lambda x:x[0])
ridge_test=T1@ridge_weights
results.append({'model':f'Ridge (alpha={ridge_alpha})','split':'validation',**ridge_val})
results.append({'model':f'Ridge (alpha={ridge_alpha})','split':'test',**physical_metrics(ridge_test,Y_test)})
predictions[f'Ridge (alpha={ridge_alpha})']=ridge_test

In [ ]:
class MLPPolicy(nn.Module):
    def __init__(self, width, depth):
        super().__init__()
        layers=[]; size=len(feature_cols)
        for _ in range(depth):
            layers += [nn.Linear(size,width),nn.ReLU()]
            size=width
        layers += [nn.Linear(size,2)]
        self.network=nn.Sequential(*layers)
    def forward(self,x): return self.network(x)

def train_mlp(width,depth,epochs=40):
    model=MLPPolicy(width,depth).to(DEVICE)
    loader=DataLoader(TensorDataset(torch.from_numpy(X_train),torch.from_numpy(Y_train)),batch_size=512,shuffle=True)
    xv=torch.from_numpy(X_val).to(DEVICE); yv=torch.from_numpy(Y_val).to(DEVICE)
    optimizer=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
    best=None; patience=8
    for epoch in range(epochs):
        model.train()
        for xb,yb in loader:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE)
            loss=nn.functional.smooth_l1_loss(model(xb),yb)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
        model.eval()
        with torch.no_grad(): val_loss=nn.functional.mse_loss(model(xv),yv).item()
        if best is None or val_loss < best[0]-1e-5:
            best=(val_loss,{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},epoch); patience=8
        else:
            patience-=1
            if patience==0: break
    model.load_state_dict(best[1]); model.eval()
    return model,best[2]+1

trained={}
for name,width,depth in [('Small MLP',64,2),('Deep MLP',128,3)]:
    model,epochs=train_mlp(width,depth)
    with torch.no_grad():
        pv=model(torch.from_numpy(X_val).to(DEVICE)).cpu().numpy()
        pt=model(torch.from_numpy(X_test).to(DEVICE)).cpu().numpy()
    results.append({'model':name,'split':'validation','epochs':epochs,**physical_metrics(pv,Y_val)})
    results.append({'model':name,'split':'test','epochs':epochs,**physical_metrics(pt,Y_test)})
    predictions[name]=pt; trained[name]=model
pd.DataFrame(results).sort_values(['split','linear_RMSE','angular_RMSE'])

In [ ]:
scores=pd.DataFrame(results)
validation=scores[scores.split=='validation'].copy()
validation['combined_normalized_RMSE']=validation.linear_RMSE/max(1e-6,train.target_linear_mps.std()) + validation.angular_RMSE/max(1e-6,train.target_angular_rps.std())
winner=validation.sort_values('combined_normalized_RMSE').iloc[0]['model']
print('Selected strictly by validation set:',winner)
display(scores[scores.model==winner])
pred=predictions[winner]*y_std+y_mean; truth=Y_test*y_std+y_mean
fig,axes=plt.subplots(2,1,figsize=(15,7),sharex=True)
n=min(1200,len(test))
axes[0].plot(truth[:n,0],label='Nav2'); axes[0].plot(pred[:n,0],label=winner,alpha=.8); axes[0].set_ylabel('linear m/s')
axes[1].plot(truth[:n,1],label='Nav2'); axes[1].plot(pred[:n,1],label=winner,alpha=.8); axes[1].set_ylabel('angular rad/s')
for ax in axes: ax.legend(); ax.grid(alpha=.2)
axes[1].set_xlabel('2 Hz test samples'); plt.tight_layout()

## Interpretation and deployment gate

Low command error means a policy imitates Nav2; it does **not** prove safe closed-loop navigation. Before replacing Nav2, replay every model in Baylands and measure mission success, duration, path length, minimum clearance, collision-monitor interventions, oscillation and reverse duration. Keep the collision monitor downstream of every learned command.

Recommended next stage:

1. Export the validation-selected MLP as the behaviour-cloning baseline.
2. Add a Gymnasium environment around the existing Baylands launcher.
3. Fine-tune with SAC (continuous linear/angular control), comparing PPO and TD3 under identical seeds.
4. Use progress, clearance and smoothness rewards; apply a large collision penalty.
5. Accept an RL policy only if it preserves 5/5 success and obstacle clearance while reducing mean mission time below the Nav2 baseline.